# Auditing pipeline

In [49]:
# Imports
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import re

In [50]:
WORKDIR = Path('/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles')

@dataclass
class PipelineConfig:
    # `workdir` is the anchor used to resolve all relative paths in the notebook.
    workdir: Path = WORKDIR

    # Local sources expected to exist in the repository.
    data_local_path: Path = Path('outputs/preprocessing/dedup_primary.tsv')
    glossary_local_path: Path = Path('data/glossary.tsv')

    # Directory where notebook exports are written.
    output_dir: Path = Path('outputs/audits')


cfg = PipelineConfig()
# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.data_path = cfg.workdir / cfg.data_local_path
cfg.glossary_path = cfg.workdir / cfg.glossary_local_path

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)
cfg

PipelineConfig(workdir=WindowsPath('/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles'), data_local_path=WindowsPath('outputs/preprocessing/dedup_primary.tsv'), glossary_local_path=WindowsPath('data/glossary.tsv'), output_dir=WindowsPath('/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles/outputs/audits'))

In [60]:
# Load data
glossary = pd.read_csv(cfg.glossary_path, sep='\t')
data = pd.read_csv(cfg.data_path, sep='\t')

# Ensure everything is lowercase for matching
glossary['surface_form'] = glossary['surface_form'].str.strip().str.lower()
glossary['target'] = glossary['target'].str.strip().str.lower()

# Pre-calculate glossary totals for denominators
# This tells us how many terms exist for every Level/Target combination
glossary_stats = glossary.groupby(['taxonomy_level', 'target']).size().to_frame('total_terms')

# Compile regex: Sorting by length (descending) prevents partial matches 
# (e.g., matching 'dog' inside 'dogwhistle')
all_forms = sorted(glossary['surface_form'].unique(), key=len, reverse=True)
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, all_forms)) + r')\b', flags=re.IGNORECASE)

In [ ]:
# 1. Extract matches
data['found_forms'] = data['text'].apply(lambda x: pattern.findall(x.lower()) if pd.notna(x) else [])

# 2. Explode and Join
# Each row in 'matches_df' represents one instance of a found dogwhistle
matches_df = data.explode('found_forms').dropna(subset=['found_forms'])

# 3. Merge with glossary to get the taxonomy_level and target for each match
# This handles cases where one surface_form might belong to multiple categories
audit_df = matches_df.merge(
    glossary, 
    left_on='found_forms', 
    right_on='surface_form', 
    how='inner'
)

Index(['post_id', 'text', 'raw_label', 'binary_hate', 'targets', 'dataset',
       'text_dedup_key', 'n_annotations', 'found_forms', 'surface_form',
       'taxonomy_level', 'type', 'target'],
      dtype='str')


In [63]:
# Build metrics aggregated by taxonomy_level and target
metrics_data = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    # Get glossary reference: all forms and types for this level/target combination
    glossary_subset = glossary[(glossary['taxonomy_level'] == level) & 
                                (glossary['target'] == target)]
    total_glossary_forms = glossary_subset['surface_form'].nunique()
    total_glossary_types = glossary_subset['type'].nunique()
    
    # Calculate metrics
    # Presence rate: proportion of distinct surface forms found vs. glossary
    distinct_forms_found = group['found_forms'].nunique()
    presence_rate = distinct_forms_found / total_glossary_forms if total_glossary_forms > 0 else 0
    
    # Type coverage: proportion of distinct dogwhistle categories (types) represented
    # Use the 'type' column already present in the group (from earlier glossary merge)
    distinct_types_found = group['type'].nunique()
    type_coverage = distinct_types_found / total_glossary_types if total_glossary_types > 0 else 0
    
    # Token frequency: total count of matched instances
    total_tokens = len(group)
    
    metrics_data.append({
        'taxonomy_level': level,
        'target': target,
        'total_glossary_forms': total_glossary_forms,
        'total_glossary_types': total_glossary_types,
        'distinct_forms_found': distinct_forms_found,
        'distinct_types_found': distinct_types_found,
        'presence_rate': presence_rate,
        'type_coverage': type_coverage,
        'token_frequency': total_tokens
    })

final_report = pd.DataFrame(metrics_data)

In [64]:
final_report

,taxonomy_level,target,total_glossary_forms,total_glossary_types,distinct_forms_found,distinct_types_found,presence_rate,type_coverage,token_frequency
0,2,african,5,1,2,1,0.400000,1.0,2
1,3,african,1,1,1,1,1.000000,1.0,42
2,3,hispanic,3,1,1,1,0.333333,1.0,1
3,4,transgender women,4,1,2,1,0.500000,1.0,3


In [65]:
# Detailed breakdown: which forms appear in each level/target group
detailed_breakdown = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    forms_list = sorted(group['found_forms'].unique())
    for form in forms_list:
        form_count = len(group[group['found_forms'] == form])
        detailed_breakdown.append({
            'taxonomy_level': level,
            'target': target,
            'surface_form': form,
            'token_count': form_count
        })

detailed_report = pd.DataFrame(detailed_breakdown).sort_values(
    by=['taxonomy_level', 'target', 'token_count'], ascending=[True, True, False]
)

print("Detailed breakdown of surface forms found:")
print(detailed_report)

Detailed breakdown of surface forms found:
   taxonomy_level             target                surface_form  token_count
0               2            african             absentee father            1
1               2            african             lack of fathers            1
2               3            african          affirmative action           42
3               3           hispanic  end birthright citizenship            1
5               4  transgender women                actual women            2
4               4  transgender women                actual woman            1


In [66]:
# Summary: Missing forms (in glossary but not found in benchmark)
missing_forms = []

for (level, target), glossary_subset in glossary.groupby(['taxonomy_level', 'target']):
    glossary_forms = set(glossary_subset['surface_form'].unique())
    found_forms = set(detailed_report[(detailed_report['taxonomy_level'] == level) & 
                                      (detailed_report['target'] == target)]['surface_form'].unique())
    missing = glossary_forms - found_forms
    
    for form in sorted(missing):
        missing_forms.append({
            'taxonomy_level': level,
            'target': target,
            'surface_form': form,
            'status': 'not_found'
        })

missing_report = pd.DataFrame(missing_forms)

print("=" * 80)
print("AUDIT SUMMARY")
print("=" * 80)
print(f"\nTotal benchmark posts analyzed: {len(data)}")
print(f"Total dogwhistle matches found: {len(audit_df)}")
print(f"Unique surface forms found: {audit_df['found_forms'].nunique()}")
print(f"\nTotal surface forms in glossary: {len(glossary)}")
print(f"Surface forms NOT found in benchmark: {len(missing_report)}")

print("\n" + "=" * 80)
print("METRICS BY TAXONOMY LEVEL & TARGET GROUP")
print("=" * 80)
print(final_report.to_string(index=False))

if len(missing_report) > 0:
    print("\n" + "=" * 80)
    print("FORMS MISSING FROM BENCHMARK")
    print("=" * 80)
    print(missing_report.to_string(index=False))

AUDIT SUMMARY

Total benchmark posts analyzed: 59621
Total dogwhistle matches found: 48
Unique surface forms found: 6

Total surface forms in glossary: 13
Surface forms NOT found in benchmark: 7

METRICS BY TAXONOMY LEVEL & TARGET GROUP
 taxonomy_level            target  total_glossary_forms  total_glossary_types  distinct_forms_found  distinct_types_found  presence_rate  type_coverage  token_frequency
              2           african                     5                     1                     2                     1       0.400000            1.0                2
              3           african                     1                     1                     1                     1       1.000000            1.0               42
              3          hispanic                     3                     1                     1                     1       0.333333            1.0                1
              4 transgender women                     4                     1          

In [67]:
# Export results
audit_metrics_path = cfg.output_dir / 'audit_metrics.tsv'
audit_detailed_path = cfg.output_dir / 'audit_detailed.tsv'
audit_missing_path = cfg.output_dir / 'audit_missing.tsv'

final_report.to_csv(audit_metrics_path, sep='\t', index=False)
detailed_report.to_csv(audit_detailed_path, sep='\t', index=False)
if len(missing_report) > 0:
    missing_report.to_csv(audit_missing_path, sep='\t', index=False)

print(f"\nExported results:")
print(f"  - {audit_metrics_path}")
print(f"  - {audit_detailed_path}")
if len(missing_report) > 0:
    print(f"  - {audit_missing_path}")


Exported results:
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\audits\audit_metrics.tsv
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\audits\audit_detailed.tsv
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\audits\audit_missing.tsv
